# 冰水主機出水溫度控制

說明出水溫度設定規則與訂定依據。

| 情境 | 措施 | 操作方式 |
| --- | --- | --- |
| S0 | 現況 | CH-1 出水約 11°C、CH-2 約 8.7~9.0°C |
| S1 | 單機時出水溫度重設 | 僅 CH-2 運轉時，依預測負荷重設出水 |
| S2 | 並聯時出水溫度重設 | 兩台並聯時，兩台設同一出水，依預測負荷重設 |

規則與耗電模型都不用氣象資料（案場拿不到天氣）。
現場兩台並聯時只能設同一出水溫度（視同一大台），負荷分配無法控制，所以不評估「用兩台設定差做最佳分配」。

## 1. 方法

- **出水規則**：出水 = 回水上限 − 當下溫差，介於下限～上限；當下溫差 = 預測負荷 ÷ (K × 流量典型值)。下限、上限、回水上限都在 `.env` 設定，單機、並聯共用
- **並聯**：兩台設同一出水；兩台共用回水，負荷依流量比例分配（回測估耗電時用該小時實測流量，控制規則不用）
- **設定與評估**：依負荷模型預測負荷決定
- **耗電模型**：CH-2 用灰箱模型；CH-1 出水溫度沒有變化過的資料，用台數控制性能曲線 × 出水溫度修正，修正沿用 CH-2 灰箱模型的敏感度（出水每變 1°C 耗電約變 1.6%）
- **資料**：9/17~10/13 逐時資料，只用穩定運轉的分鐘（排除開機後 60 分鐘、停機前 10 分鐘）；冰水泵固定頻率運轉，各情境流量相同，泵耗電不列入比較

## 2. 環境設定與模型

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# 切到專案根目錄，才 import 得到 model、research
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from data.sensor import is_running, load_chiller
from model import rules
from research import ch2_model as ch2
from research import chiller
from research.ch2_model import LOAD_LIMIT_RT, WARMUP_MIN
from research.chart_style import show

pio.renderers.default = "jupyterlab"

TZ = "Asia/Taipei"
START = pd.Timestamp("2026-09-17", tz=TZ)
END = pd.Timestamp("2026-10-14", tz=TZ)
SHUTDOWN_MIN = 10  # 停機前 10 分鐘為暫態，排除
MIN_STEADY = 45  # 每小時至少穩定運轉 45 分鐘才評估

In [2]:
# 模型：台數控制性能曲線、CH-2 灰箱耗電模型
curves = await chiller.load_curves()
trainer = ch2.PowerModelTrainer()
await trainer.main()
ch2_model = trainer.model
# 出水每調高 1°C，耗電減少的比例
sensitivity = -ch2.change_pct(ch2_model, trainer.df, 1.0) / 100

# CH-2 模型的資料範圍（超出為外推）；滿載設定：現場平常的出水設定（CH-2 負荷 ≥ LOAD_LIMIT_RT 時出水第 10 百分位）
T_RANGE = (trainer.df["T_CHWS"].min(), trainer.df["T_CHWS"].max())
Q_RANGE = (ch2.cooling_rt(trainer.df).min(), ch2.cooling_rt(trainer.df).max())
ch2_raw = await load_chiller(rules.CHILLERS["CH-2"]["id"])
# 資料推估的滿載設定（比對 .env 的出水下限用）
T_DATA = chiller.full_load_supply(ch2_raw)
print(
    f"CH-2 灰箱模型：CV(RMSE) {trainer.metric['CV(RMSE) %']:.1f}%，"
    f"出水每調高 1°C 耗電約 −{sensitivity * 100:.2f}%；"
    f"建模範圍 出水 {T_RANGE[0]:.1f}~{T_RANGE[1]:.1f}°C、"
    f"負荷 {Q_RANGE[0]:.0f}~{Q_RANGE[1]:.0f} RT"
)
print(
    f".env 規則參數：出水下限 {rules.T_MIN}°C、上限 {rules.T_MAX}°C、"
    f"回水上限 {rules.T_RETURN}°C、並聯門檻 {rules.PARALLEL_ABOVE} RT"
)
print(
    f"資料推估的滿載設定 {T_DATA:.2f}°C"
    f"（CH-2 負荷 ≥ {LOAD_LIMIT_RT} RT 時出水第 {chiller.T_FULL_PCT} 百分位），"
    f"可與出水下限比對"
)

CH-2 灰箱模型：CV(RMSE) 3.1%，出水每調高 1°C 耗電約 −1.64%；建模範圍 出水 8.5~9.8°C、負荷 218~361 RT
.env 規則參數：出水下限 8.7°C、上限 12.0°C、回水上限 13.9°C、並聯門檻 380 RT
資料推估的滿載設定 8.70°C（CH-2 負荷 ≥ 360 RT 時出水第 10 百分位），可與出水下限比對


In [3]:
# 穩定運轉的分鐘：排除開機後 WARMUP_MIN 分鐘、停機前 SHUTDOWN_MIN 分鐘
def steady_minutes(minutes: pd.DataFrame) -> pd.Series:
    running = is_running(minutes)
    segment = (running != running.shift()).cumsum()  # 每次開機或停機就換一段
    run_min = running.groupby(segment).cumsum()  # 已運轉幾分鐘
    left_min = running.groupby(segment).transform("sum") - run_min  # 離停機還有幾分鐘
    return (
        running
        & (run_min > WARMUP_MIN)
        & (left_min >= SHUTDOWN_MIN)
        & (minutes["Q_COOL"] > 50)
    )


# 逐時資料：只取穩定運轉分鐘的平均；欄位尾碼 1 = CH-1、2 = CH-2
parts = []
minute_data = {}
for number, chiller_config in enumerate(rules.CHILLERS.values(), start=1):
    minutes = (await load_chiller(chiller_config["id"])).resample("1min").mean()
    steady = steady_minutes(minutes)
    # 每分鐘資料（開停機的小時分段用）
    minute_data[number] = minutes.assign(STEADY=steady, ON=is_running(minutes))
    steady_avg = minutes[steady].resample("1h").mean()
    parts.append(
        pd.DataFrame(
            {
                f"Q{number}": steady_avg["Q_COOL"],
                f"T{number}": steady_avg["T_CHWS"],
                f"R{number}": steady_avg["T_CHWR"],
                f"F{number}": steady_avg["FLOW"],
                f"KW{number}": steady_avg["kW"],
                # 穩定運轉滿 MIN_STEADY 分鐘
                f"ON{number}": steady.resample("1h").sum() >= MIN_STEADY,
                # 整小時停機
                f"OFF{number}": is_running(minutes).resample("1h").sum() == 0,
            }
        )
    )
    # 時序圖用（含開停機的小時）：負荷、出水取運轉分鐘平均；整小時負荷、耗電停機的分鐘記 0
    running = is_running(minutes)
    running_avg = minutes[running].resample("1h").mean()
    hour_avg = minutes[["Q_COOL", "kW"]].where(running, 0).resample("1h").mean()
    parts.append(
        pd.DataFrame(
            {
                f"QR{number}": running_avg["Q_COOL"],
                f"TR{number}": running_avg["T_CHWS"],
                f"QH{number}": hour_avg["Q_COOL"],
                f"KWH{number}": hour_avg["kW"],
                f"RUN{number}": running.resample("1h").sum() > 0,
            }
        )
    )
hour_range = pd.date_range(START, END, freq="1h", inclusive="left")
hourly = pd.concat(parts, axis=1).reindex(hour_range)
for number in (1, 2):  # 沒有資料的小時：不算穩定運轉、算停機
    hourly[f"ON{number}"] = hourly[f"ON{number}"].eq(True)
    hourly[f"OFF{number}"] = hourly[f"OFF{number}"].ne(False)
    hourly[f"RUN{number}"] = hourly[f"RUN{number}"].eq(True)

# 只評估「兩台並聯」與「僅 CH-2」穩定運轉的小時
evaluable = (hourly.ON1 & hourly.ON2) | (hourly.ON2 & hourly.OFF1)
hourly["ON1"] &= evaluable
hourly["ON2"] &= evaluable
# 實測耗電
ch1_meter = hourly["KW1"].where(hourly.ON1, 0)
ch2_meter = hourly["KW2"].where(hourly.ON2, 0)
hourly["METER"] = ch1_meter + ch2_meter
parallel_count = (hourly.ON1 & hourly.ON2).sum()
ch2_only_count = (hourly.ON2 & ~hourly.ON1).sum()
print(
    f"可評估 {evaluable.sum()} 小時"
    f"（並聯 {parallel_count}、僅 CH-2 {ch2_only_count}）"
)

# 規則用的流量典型值（同 control.typical_flows：穩定運轉小時的中位數）：只開 CH-2 用 CH-2 流量、並聯用兩台合計
FLOWS = {
    "single": float(hourly.F2[hourly.ON2 & ~hourly.ON1].median()),
    "parallel": float((hourly.F1 + hourly.F2)[hourly.ON1 & hourly.ON2].median()),
}
print(
    f"流量典型值：只開 CH-2 {FLOWS['single']:,.0f} LPM、"
    f"兩台並聯合計 {FLOWS['parallel']:,.0f} LPM"
)

可評估 103 小時（並聯 20、僅 CH-2 83）
流量典型值：只開 CH-2 3,852 LPM、兩台並聯合計 6,088 LPM


In [4]:
# 時序圖用：只排有資料的小時，不同天之間插一格空白（線不跨天相連）；回傳 (橫軸標籤, 資料)
def by_day(df: pd.DataFrame) -> tuple[list, pd.DataFrame]:
    parts = []
    labels = []
    day_groups = df.groupby(df.index.strftime("%m/%d"), sort=False)
    for i, (day, group) in enumerate(day_groups):
        if i > 0:
            parts.append(pd.DataFrame(np.nan, index=[0], columns=df.columns))
            labels.append(f"｜{day}")
        parts.append(group.reset_index(drop=True))
        labels += list(group.index.strftime("%m/%d %H:00"))
    return labels, pd.concat(parts, ignore_index=True)


# 類別橫軸刻度標每天第一個小時的日期；相鄰刻度太近（運轉時數少的日子）就略過，最多約 n_max 個
def hour_axis(fig: go.Figure, labels: list, n_max: int = 12) -> None:
    # 每天第一個小時的位置（空白格之後的第一格）
    day_starts = []
    for i, tick_label in enumerate(labels):
        if tick_label.startswith("｜"):
            continue
        if i == 0 or labels[i - 1].startswith("｜"):
            day_starts.append(i)
    keep = []
    last = -np.inf
    for i in day_starts:
        if i - last >= len(labels) / n_max:
            keep.append(i)
            last = i
    fig.update_xaxes(
        type="category",
        tickvals=[labels[i] for i in keep],
        ticktext=[labels[i][:5] for i in keep],
        tickangle=0,
    )

## 3. 單機出水溫度重設規則（S1）

### 3.1 設定公式與參數

只開 CH-2 時，依負荷算出出水設定：

**出水 = 13.9°C − 預測負荷 ÷ (K × 流量)**，最高 12°C、最低 8.7°C

K = 4.186 ÷ 60 ÷ 3.517（水的比熱、LPM 換算為每秒、kW 換算為 RT），K × 流量 = 每 1°C 溫差可帶走的冷量 (RT/°C)

- 8.7°C：出水下限（`.env` 的 `CONTROL_T_MIN`，現場平常的出水設定；資料推估的滿載設定也是 8.70°C）
- 13.9°C：回水上限（`.env` 的 `CONTROL_T_RETURN`），為 CH-2 額定 400 RT、出水 8.7°C 時的回水。只開 CH-2 時流量典型值 3,852 LPM，每 1°C 溫差可帶走約 76 RT（K × 流量），8.7 + 400 ÷ 76 ≈ 13.9°C，即負荷達額定時才降到出水下限
- 12°C：出水上限（`.env` 的 `CONTROL_T_MAX`，PM 試行各階段改這個值）

負荷越小，溫差越小，出水就可以調高，讓回水維持在 13.9°C；負荷越大，出水要越冷，回水才不會變熱、末端才供得上冷。例如 300 RT 時設 10.0°C、200 RT 時設 11.3°C；負荷低於約 145 RT 時設為上限 12°C，高於約 397 RT 時為下限 8.7°C。

規則沒有考慮外氣濕度（案場拿不到天氣資料），潮濕的日子出水調高後除濕能力會下降，試行時需注意室內濕度。

### 3.2 規則與實測出水

下圖紅線為規則：水平段為上限 12°C，斜線段由負荷決定。
藍點為只開 CH-2 穩定運轉各小時的實際出水；負荷 382~396 RT 時有 10 小時出水高於 9.2°C，主機已滿載、無法降到設定值。

In [5]:
alone = hourly[hourly.ON2 & ~hourly.ON1]  # 只開 CH-2 且穩定運轉的小時
RATED2 = curves["CH-2"]["rated"]
F_ALONE = FLOWS["single"]  # 只開 CH-2 時的冰水流量典型值
KF = rules.K * F_ALONE  # 每 1°C 溫差可帶走的冷 (RT)


# 只開 CH-2 時規則建議的出水設定
def alone_setpoint(load_rt):
    return rules.supply_target(load_rt, F_ALONE)


drift = alone.T2 > rules.T_MIN + 0.5  # 出水拉不下來
loads = np.arange(100, 421, 5)
EX = [300, 200]  # 說明用的例子：負荷 RT
Q_CAP = (rules.T_RETURN - rules.T_MAX) * KF  # 低於這個負荷時規則已達上限 T_MAX

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=loads,
        y=alone_setpoint(loads),
        mode="lines",
        name="建議設定",
        legend="legend",
        line=dict(color="#a33f17", width=2.5),
        hovertemplate="負荷 %{x} RT｜出水 %{y:.1f}°C<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=alone.Q2,
        y=alone.T2,
        mode="markers",
        name="只開 CH-2、穩定運轉的小時",
        legend="legend2",
        marker=dict(size=6, color="#86b6ef", opacity=0.8),
        text=alone.index.strftime("%m/%d %H:00"),
        hovertemplate="%{text}｜負荷 %{x:.0f} RT｜出水 %{y:.2f}°C<extra></extra>",
    )
)
example_load = EX[0]
fig.add_trace(
    go.Scatter(
        x=[example_load],
        y=[float(alone_setpoint(example_load))],
        mode="markers",
        showlegend=False,
        marker=dict(
            size=13,
            color="#a33f17",
            symbol="star",
            line=dict(width=1, color="white"),
        ),
        hovertemplate=(
            f"例：{example_load} RT → "
            f"建議 {float(alone_setpoint(example_load)):.1f}°C<extra></extra>"
        ),
    )
)
# 標註：(x, y, 文字, 箭頭 x 偏移, 箭頭 y 偏移)
annotations = [
    (130, rules.T_MAX, f"水平段：上限 {rules.T_MAX:g}°C", 0, -28),
    (
        350,
        float(alone_setpoint(350)),
        f"斜線段：由負荷決定（回水上限 {rules.T_RETURN}°C）",
        -30,
        -90,
    ),
    (
        (rules.T_RETURN - rules.T_MIN) * KF,
        rules.T_MIN,
        f"下限 {rules.T_MIN:.1f}°C",
        40,
        45,
    ),
    (
        example_load,
        float(alone_setpoint(example_load)),
        f"例：{example_load} RT → {float(alone_setpoint(example_load)):.1f}°C",
        -70,
        60,
    ),
]
for note_x, note_y, text, arrow_x, arrow_y in annotations:
    fig.add_annotation(
        x=note_x,
        y=note_y,
        text=text,
        ax=arrow_x,
        ay=arrow_y,
        showarrow=True,
        arrowhead=0,
        arrowcolor="#52514e",
        font=dict(size=13, color="#52514e"),
        bgcolor="rgba(255,255,255,0.85)",
    )
fig.add_vline(
    x=rules.PARALLEL_ABOVE,
    line=dict(color="#8a8984", width=1.5, dash="dash"),
)
fig.add_annotation(
    x=rules.PARALLEL_ABOVE,
    y=12.4,
    text=f"超過 {rules.PARALLEL_ABOVE} RT 改開兩台",
    showarrow=False,
    xanchor="right",
    font=dict(size=12, color="#52514e"),
)
fig.update_xaxes(title_text="CH-2 負荷 (RT)")
fig.update_yaxes(title_text="CH-2 出水溫度 (°C)", range=[7.5, 12.5])
show(
    fig,
    "只開 CH-2 時的出水設定規則",
    f"出水 = {rules.T_RETURN}°C − 負荷 ÷ {KF:.0f} RT/°C，"
    f"介於 {rules.T_MIN:.1f}～{rules.T_MAX:g}°C",
    legend_groups=["建議設定", "現況出水"],
    height=700,
)

print(
    f"只開 CH-2 穩定運轉 {len(alone)} 小時；"
    f"流量典型值 {F_ALONE:,.0f} LPM，每 1°C 溫差約 {KF:.0f} RT；"
    f"回水上限 {rules.T_RETURN}°C；"
    f"負荷低於 {Q_CAP:.0f} RT 時設定為上限 {rules.T_MAX:g}°C、"
    f"高於 {(rules.T_RETURN - rules.T_MIN) * KF:.0f} RT 時為下限 {rules.T_MIN:.1f}°C"
)
for load_rt in EX:
    print(
        f"例：{load_rt} RT → {rules.T_RETURN} − {load_rt} ÷ {KF:.0f} = "
        f"{float(alone_setpoint(load_rt)):.1f}°C"
    )
drift_hours = alone[drift]
print(
    f"出水拉不下來（> {rules.T_MIN + 0.5:.1f}°C）{len(drift_hours)} 小時："
    f"CH-2 負荷 {drift_hours.Q2.min():.0f}~{drift_hours.Q2.max():.0f} RT、"
    f"出水 {drift_hours.T2.min():.1f}~{drift_hours.T2.max():.1f}°C、"
    f"回水 {drift_hours.R2.min():.1f}~{drift_hours.R2.max():.1f}°C"
    f"（其他小時回水中位數 {alone.R2[~drift].median():.1f}°C）、"
    f"耗電 {drift_hours.KW2.min():.0f}~{drift_hours.KW2.max():.0f} kW"
    f"（只開 CH-2 小時最大 {alone.KW2.max():.0f} kW）"
)
print(
    f"回水上限 {rules.T_RETURN}°C ≈ 出水下限 {rules.T_MIN}°C + "
    f"額定 {RATED2} RT ÷ {KF:.0f} RT/°C = {rules.T_MIN + RATED2 / KF:.2f}°C"
)

只開 CH-2 穩定運轉 83 小時；流量典型值 3,852 LPM，每 1°C 溫差約 76 RT；回水上限 13.9°C；負荷低於 145 RT 時設定為上限 12°C、高於 397 RT 時為下限 8.7°C
例：300 RT → 13.9 − 300 ÷ 76 = 10.0°C
例：200 RT → 13.9 − 200 ÷ 76 = 11.3°C
出水拉不下來（> 9.2°C）10 小時：CH-2 負荷 382~396 RT、出水 9.2~10.5°C、回水 14.0~15.1°C（其他小時回水中位數 12.5°C）、耗電 233~252 kW（只開 CH-2 小時最大 252 kW）
回水上限 13.9°C ≈ 出水下限 8.7°C + 額定 400 RT ÷ 76 RT/°C = 13.93°C


## 4. 現況：並聯時負荷分配不均

並聯時 CH-1 出水固定約 11°C、CH-2 約 9°C，CH-2 出水較冷而承擔大部分負荷、常接近滿載；CH-1 只承擔少部分負荷、在低效率區運轉。

In [6]:
parallel_hours = hourly[hourly.ON1 & hourly.ON2]  # 並聯的小時
q_total = parallel_hours.Q1 + parallel_hours.Q2
print(
    f"並聯 {len(parallel_hours)} 小時，"
    f"總負荷 {q_total.min():.0f}~{q_total.max():.0f} RT"
)
for name, number in [("CH-1", 1), ("CH-2", 2)]:
    mean_load = parallel_hours[f"Q{number}"].mean()
    mean_supply = parallel_hours[f"T{number}"].mean()
    load_ratio = mean_load / curves[name]["rated"] * 100
    print(
        f"  {name}：出水平均 {mean_supply:.1f}°C、"
        f"負荷平均 {mean_load:.0f} RT（負載率 {load_ratio:.0f}%）"
    )
print(f"  CH-2 負荷占比平均 {(parallel_hours.Q2 / q_total).mean() * 100:.0f}%")

並聯 20 小時，總負荷 396~525 RT
  CH-1：出水平均 11.0°C、負荷平均 144 RT（負載率 38%）
  CH-2：出水平均 9.0°C、負荷平均 307 RT（負載率 77%）
  CH-2 負荷占比平均 68%


In [7]:
both_run = hourly[hourly.RUN1 & hourly.RUN2]  # 兩台都有開的小時（含開停機）
both_run_columns = both_run[["QR1", "QR2", "TR1", "TR2"]].set_axis(
    ["Q1", "Q2", "T1", "T2"], axis=1
)
tick_labels, plot_data = by_day(both_run_columns)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12,
    subplot_titles=("各機負荷 (RT)", "各機出水溫度 (°C)"),
)
for name, number, color in [("CH-1", 1, "#2a78d6"), ("CH-2", 2, "#eb6834")]:
    fig.add_trace(
        go.Scatter(
            x=tick_labels,
            y=plot_data[f"Q{number}"],
            mode="lines+markers",
            name=name,
            line=dict(color=color, width=2.5),
            marker=dict(size=5),
            hovertemplate=f"{name} %{{y:.0f}} RT<extra></extra>",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=tick_labels,
            y=plot_data[f"T{number}"],
            mode="lines+markers",
            name=name,
            showlegend=False,
            line=dict(color=color, width=2.5),
            marker=dict(size=5),
            hovertemplate=f"{name} %{{y:.2f}}°C<extra></extra>",
        ),
        row=2,
        col=1,
    )
hour_axis(fig, tick_labels)
fig.update_layout(hovermode="x unified")
show(
    fig,
    "並聯時各機負荷與出水溫度",
    "兩台都有開的小時（含開停機的小時）；以運轉分鐘平均",
    height=700,
)

## 5. 控制規則

依預測負荷決定開機台數與出水設定：

- **總負荷 ≤ 380 RT：僅開 CH-2**，出水依第 3 節規則
- **總負荷 > 380 RT：兩台並聯**，兩台設同一出水，用同一條規則，流量改用兩台合計（約 628 RT 以上為下限 8.7°C）
- 流量：用穩定運轉時的典型值（只開 CH-2 約 3,852 LPM、並聯合計約 6,088 LPM）；上線時依最近 14 天監控資料自動更新

In [8]:
# 控制規則：各總負荷下的運轉方式與出水設定（並聯時兩台同一設定）
LOADS = np.r_[
    np.arange(150.0, 701, 25), rules.PARALLEL_ABOVE, rules.PARALLEL_ABOVE + 1
].astype(float)
LOADS.sort()
rule_table = rules.control_rule(LOADS, FLOWS).set_index("全館負荷 RT")

fig = go.Figure(
    go.Scatter(
        x=LOADS,
        y=rule_table["出水設定 °C"],
        mode="lines+markers",
        name="出水設定",
        line=dict(color="#a33f17", width=2.5),
        marker=dict(size=6),
        customdata=rule_table["建議方案"],
        hovertemplate=(
            "總負荷 %{x:.0f} RT｜%{customdata}｜出水 %{y:.1f}°C<extra></extra>"
        ),
    )
)
# 並聯區
fig.add_vrect(
    x0=rules.PARALLEL_ABOVE,
    x1=LOADS.max(),
    fillcolor="#e8e6e1",
    opacity=0.5,
    line_width=0,
    layer="below",
)
for note_x, text in [(275, "僅開 CH-2"), (550, "兩台並聯（兩台同一設定）")]:
    fig.add_annotation(
        x=note_x,
        y=12.3,
        text=text,
        showarrow=False,
        font=dict(size=13, color="#52514e"),
    )
fig.update_xaxes(title_text="總負荷 (RT)")
fig.update_yaxes(title_text="出水溫度設定 (°C)", range=[7.5, 12.6])
fig.update_layout(showlegend=False)
show(
    fig,
    "出水溫度控制規則",
    f"流量典型值：只開 CH-2 {FLOWS['single']:,.0f} LPM、"
    f"並聯合計 {FLOWS['parallel']:,.0f} LPM；"
    f"出水 = {rules.T_RETURN}°C − 負荷 ÷ (K × 流量)",
    height=650,
)

## 6. 出水溫度控制規則總結與限制

**控制規則**
1. **台數**：依 onoff_control.ipynb 規則（預測冷負荷 < 60 RT 停機、60~380 RT 單機 CH-2、> 380 RT 兩台並聯，含最短運轉時段）
2. **出水設定**：出水 = 13.9°C − 預測負荷 ÷ (K × 流量典型值)，限制於 8.7~12°C；負荷越低，出水設定越高
   - 8.7°C 出水下限（`CONTROL_T_MIN`）：現場平常的出水設定
   - 13.9°C 回水上限（`CONTROL_T_RETURN`）：CH-2 額定 400 RT、出水 8.7°C 時的回水
   - 12°C 出水上限（`CONTROL_T_MAX`）：PM 試行各階段調整
3. **單機（CH-2）**：流量用只開 CH-2 時的典型值；約 145 RT 以下為上限 12°C，約 397 RT 以上為下限 8.7°C（第 3 節）
4. **並聯**：兩台設同一出水，流量用兩台合計的典型值；負荷依流量比例分配，CH-1 多分擔負荷、離開低效率區（第 4 節）
5. **流量典型值**：最近 14 天穩定運轉小時的中位數，上線時自動更新
6. 規則不使用氣象資料

**限制**
- 回水上限 13.9°C 沒有室內溫濕度資料可佐證；規則不看外氣濕度，潮濕時除濕能力可能不足
- 規則以預測負荷和流量典型值估算回水；實測冰水流量會隨負荷變化，加上負荷預測誤差，實際回水會在 13.9°C 上下浮動，回測中約一半時段略高（見 control_backtest.ipynb 第 6 節）
- CH-1 出水只在 11°C 附近運轉過，出水影響沿用 CH-2 敏感度估計；若 CH-1 實際更敏感，並聯同設定的效益會較少，待 CH-1 型錄或試行驗證
- CH-2 模型建模範圍為出水 8.5~9.8°C，超出為外推；CH-2 模型不含外氣影響